# Qwen3.6-35B-A3B — Distillation SFT on a free Colab T4

Trains **Qwen3.6-35B-A3B** (hybrid MoE: gated-deltanet + full attention, 256 experts)
on a mix of:

- `freakyskittle/glm-5.3-distillation` *(private — uses your Colab `HF_TOKEN` secret)*
- security agent traces: `ciTy1504/security-agentic-sft-corpus` (cybench, autopenbench, CVE factory)
- R1-distilled vulnerability reasoning: `Leopo1d/OpenVul_*`, `SecCoderX`
- CTF solver traces: `TrueNix/ctf-solver-dataset`
- GLM-5.1 SWE-agent traces: `whitecircle/swe-rebench-v2-glm-5.1-*`
- benchmark train splits: GSM8K, MATH, ARC-Challenge, MMLU

> **Checkpoint note.** `HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive` is
> **GGUF-only** and cannot be trained directly. This notebook defaults to
> `huihui-ai/Huihui-Qwen3.6-35B-A3B-abliterated` — abliterated safetensors, the
> closest trainable equivalent. Set `MODEL_NAME` to `unsloth/Qwen3.6-35B-A3B` or
> `Qwen/Qwen3.6-35B-A3B` to train the official weights instead.
>
> **T4 reality — verified on a real T4 (Colab high-RAM shape).**
> 4-bit weights are ~19 GB > 16 GB VRAM, so `offload_layers` swaps decoder
> layers to host RAM. A standard 12 GB-RAM T4 **cannot** host-swap this model —
> the load transient alone exceeds the cgroup limit; use Runtime > Change
> runtime type > T4 **high-RAM** (~51 GB). Attention-only LoRA is required:
> targeting expert `gate/up/down_proj` creates ~30k adapters (~9 GB) and OOMs.
> T4 has no native bf16 — this model NaNs in fp16, so Unsloth loads emulated
> bf16 + fp32 kernels: ~3–6 min/step. A 3-step smoke (E2E verified: load ->
> adapter -> data -> steps -> save) is the realistic T4 goal; do real epochs on
> an L4/A100/H100.


In [ ]:
#@title 1. Environment + GPU check
import torch
print(torch.__version__, torch.cuda.is_available())
print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
#@title 2. Install Unsloth (from the fork) + deps
%%capture
!pip install --upgrade --no-deps "git+https://github.com/Jackson57279/unsloth.git"
!pip install --upgrade unsloth_zoo trl peft bitsandbytes accelerate datasets \
    "transformers>=5.2.0" "huggingface_hub" hf_transfer sentencepiece
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [ ]:
#@title 3. HF login (needed for the private dataset)
# Colab: add a secret named HF_TOKEN in the key panel, or paste it below.
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token = userdata.get("HF_TOKEN"))
except Exception:
    login()  # uses HF_TOKEN env var or interactive prompt
print("logged in")

In [ ]:
#@title 4. Config
MODEL_NAME     = "huihui-ai/Huihui-Qwen3.6-35B-A3B-abliterated"  # abliterated ~= the linked GGUF
# MODEL_NAME   = "unsloth/Qwen3.6-35B-A3B"   # official base mirror
MAX_SEQ        = 2048        # raise to 4096 if VRAM allows (traces are long)
OFFLOAD_LAYERS = 24           # host-swap layers; needs high-RAM T4 (~51GB host)
LORA_TARGETS   = "attn"     # "attn" fits T4; "moe" adds ~30k expert adapters (>30GB VRAM)
LORA_R         = 32
LORA_ALPHA     = 64
LR             = 2e-4
EPOCHS         = 1
MAX_STEPS      = -1          # set e.g. 3 for a smoke run
SMOKE          = False       # True = 64 rows/source + 3 steps
PER_SOURCE_CAP = None        # e.g. 500 to shrink every dataset
RUN_EVAL       = True        # bounded ARC/MMLU/GSM8K eval after training
PUSH_TO_HUB    = None        # e.g. "freakyskittle/qwen36-distill-lora"


In [ ]:
#@title 5. Load model (text-only, 4-bit, layer offload)
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ,
    dtype = None,             # Unsloth picks: forced bf16 on T4 for this family
    load_in_4bit = True,
    text_only = True,         # drops the vision tower
    offload_layers = OFFLOAD_LAYERS,
)
print("loaded:", model.dtype)
!nvidia-smi --query-gpu=memory.used,memory.free --format=csv

In [ ]:
#@title 6. LoRA (MoE experts are auto-targeted by Unsloth)
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]
if LORA_TARGETS == "moe":
    target_modules += ["gate_proj", "up_proj", "down_proj"]  # ~30k adapters!

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = target_modules,
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable params: {trainable/1e6:.1f}M")


In [ ]:
#@title 7. Fetch the pipeline modules from the fork
for _f in ("data_prep.py", "eval_bench.py", "model_card.py"):
    url = f"https://raw.githubusercontent.com/Jackson57279/unsloth/main/examples/distillation_sft/{_f}"
    import urllib.request; urllib.request.urlretrieve(url, _f)
import data_prep
print("sources:", len(data_prep.DATASET_SPECS))

In [ ]:
#@title 8. Build the mixed dataset
specs = data_prep.DATASET_SPECS
if SMOKE:
    MAX_STEPS = 3
    specs = [{**s, "max_rows": min(s.get("max_rows", 64), 64), "slice": True} for s in specs]
elif PER_SOURCE_CAP:
    specs = [{**s, "max_rows": min(s.get("max_rows", PER_SOURCE_CAP), PER_SOURCE_CAP)} for s in specs]

ds = data_prep.build_chat_dataset(
    tokenizer,
    specs = specs,
    max_seq_length = MAX_SEQ,
)
print(ds)
print("----- SAMPLE -----")
print(ds[0]["text"][:1200])


In [ ]:
#@title 9. Train
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

use_bf16 = model.dtype == torch.bfloat16 or is_bfloat16_supported()
print("training dtype:", "bf16" if use_bf16 else "fp16")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds,
    args = SFTConfig(
        dataset_text_field = "text",
        max_length = MAX_SEQ,
        dataset_num_proc = 1,   # forking CoW-duplicates the resident model -> OOM
        packing = True,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,
        num_train_epochs = EPOCHS,
        max_steps = MAX_STEPS,
        learning_rate = LR,
        lr_scheduler_type = "cosine",
        warmup_ratio = 0.05,
        weight_decay = 0.01,
        max_grad_norm = 1.0,
        optim = "adamw_8bit",
        bf16 = use_bf16,
        fp16 = not use_bf16,
        logging_steps = 1,
        save_strategy = "steps",
        save_steps = 100,
        save_total_limit = 2,
        output_dir = "outputs/qwen36-distill",
        seed = 3407,
        report_to = "none",
    ),
)

stats = trainer.train()
print(stats.metrics)


In [ ]:
#@title 10. Benchmark eval + sanity generations
import json, os
results = {}
if RUN_EVAL:
    import eval_bench
    results = eval_bench.run_eval(model, tokenizer)
    os.makedirs("outputs/qwen36-distill", exist_ok = True)
    with open("outputs/qwen36-distill/results.json", "w") as f:
        json.dump(results, f, indent = 2)
    print(json.dumps({k: v for k, v in results.items() if k != "security_qualitative"}, indent = 2))
else:
    FastLanguageModel.for_inference(model)
    msgs = [{"role": "user", "content":
             "Audit this C function for vulnerabilities:\n\n"
             "void copy(char *dst, const char *src) { while (*src) *dst++ = *src++; *dst = 0; }"}]
    inputs = tokenizer.apply_chat_template(msgs, tokenize = True,
                                         add_generation_prompt = True,
                                         return_tensors = "pt").to("cuda")
    out = model.generate(inputs, max_new_tokens = 256, temperature = 0.6, top_p = 0.9)
    print(tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens = False))

In [ ]:
#@title 11. Save adapter + model card (+ optional push / GGUF export)
model.save_pretrained("outputs/qwen36-distill/lora")
tokenizer.save_pretrained("outputs/qwen36-distill/lora")

import model_card
card = model_card.build_model_card(
    repo_id = PUSH_TO_HUB or "qwen36-distill-lora",
    base_model = MODEL_NAME,
    results_path = "outputs/qwen36-distill/results.json" if RUN_EVAL else None,
    train_stats = dict(stats.metrics),
    dataset_specs = specs,
    hyperparams = {
        "method": "QLoRA SFT (4-bit base, LoRA adapters)",
        "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
        "learning_rate": LR, "max_seq_length": MAX_SEQ,
        "epochs": EPOCHS, "packing": True,
        "offload_layers": OFFLOAD_LAYERS,
    },
    hardware = torch.cuda.get_device_name(0),
)
with open("outputs/qwen36-distill/lora/README.md", "w") as f:
    f.write(card)

if PUSH_TO_HUB:
    model.push_to_hub(PUSH_TO_HUB)
    tokenizer.push_to_hub(PUSH_TO_HUB)
    print("pushed:", PUSH_TO_HUB)

# To ship a merged+quantized GGUF (needs a beefier machine / more disk):
# model.save_pretrained_gguf("outputs/qwen36-distill/gguf", tokenizer, quantization_method = "q4_k_m")